# M03-02 — Reglas de negocio

Referencia de validación. El alumno trabaja en `notebooks/alumno/M03-02-reglas-negocio.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, when, lower, least, lit
spark = get_spark("novashop-m03")
lines = spark.read.parquet(str(STAGING / "lines_enriched"))
fact = (
    lines.withColumn("discount", least(col("discount"), lit(1.0)))
    .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel"))).otherwise(lit("other")),
    )
    .withColumn("is_billable", col("status") == "paid")
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
)
fact.groupBy("channel_norm").count().orderBy("channel_norm").show()
assert fact.count() == 1980
assert fact.where(col("gmv_line") < 0).count() == 0
assert fact.where(col("discount") > 1).count() == 0
assert fact.where(col("is_billable")).count() == 1127
chans = {r.channel_norm for r in fact.select("channel_norm").distinct().collect()}
assert chans <= {"web", "app", "store", "other"}
fact.write.mode("overwrite").parquet(str(STAGING / "fact_lines"))
assert spark.read.parquet(str(STAGING / "fact_lines")).count() == 1980
print("M03-02 OK")
